In [ ]:
import { RecursiveSet as Set, Tuple, Value, flatMap } from 'recursive-set';

In [ ]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [ ]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

# The [Zebra Puzzle](https://en.wikipedia.org/wiki/Zebra_Puzzle)

The following puzzle appeared in the magazine *Life International* on the 17th of December in the year 1962:
* There are five houses.
* The Englishman lives in the red house.
* The Spaniard owns the dog.
* Coffee is drunk in the green house.
* The Ukrainian drinks tea.
* The green house is immediately to the right of the ivory house.
* The Old Gold smoker owns snails.
* Kools are smoked in the yellow house.
* Milk is drunk in the middle house.
* The Norwegian lives in the first house.
* The man who smokes Chesterfields lives in the house next to the man with the fox.
* Kools are smoked in the house next to the house where the horse is kept.
* The Lucky Strike smoker drinks orange juice.
* The Japanese smokes Parliaments.
* The Norwegian lives next to the blue house.

Furthermore, each of the five houses is painted in a different colour, their inhabitants are of different nationalities, own different pets, drink different beverages, and smoke different brands of cigarettes.

Our task is to answers the following questions: 
1. *Who drinks water?*
2. *Who owns the zebra?*

## Choosing the Appropriate Variables

In order to solve this problem we use the following propositional variables, where the counter $i$ ranges from $1$ to $5$.  

* $\texttt{English}\langle i \rangle$ expresses the fact
  that the Englishman lives in house number $i$.
  
  The remaining nationalities are coded using the variables
  $\texttt{Spanish}\langle i \rangle$, $\texttt{Ukrainian}\langle i \rangle$,
  $\texttt{Norwegian}\langle i \rangle$, and $\texttt{Japanese}\langle i \rangle$.
* $\mathtt{Red}\langle i \rangle$ expresses that house number $i$ is red.
  
  We use the variables $\texttt{Green}\langle i \rangle$, $\texttt{Ivory}\langle i \rangle$,
  $\texttt{Yellow}\langle i \rangle$, and $\texttt{Blue}\langle i \rangle$ to encode the remaining
  colors.
* $\texttt{OldGold}\langle i \rangle$ expresses that the inhabitant of house number $i$ smokes cigarettes of the brand "Old Gold".
  
  We use the variables $\texttt{Kools}\langle i \rangle$,
  $\texttt{Chesterfields}\langle i \rangle$, $\texttt{LuckyStrike}\langle i \rangle$, and
  $\texttt{Parliaments}\langle i \rangle$ to encode the cigarette brands.
* $\texttt{Dog}\langle i \rangle$ expresses the fact that the inhabitant of house number
  $i$ keeps a dog as his pet.

  We use the variables $\texttt{Snails}\langle i \rangle$, $\texttt{Fox}\langle i \rangle$,
  $\texttt{Horse}\langle i \rangle$, and $\texttt{Zebra}\langle i \rangle$ to encode the
  remaining pets.
* $\texttt{Coffee}\langle i \rangle$ expresses the fact the the inhabitant of house number $i$
  drinks coffee.

  We use the variables $\texttt{Milk}\langle i \rangle$,
  $\texttt{OrangeJuice}\langle i \rangle$, $\texttt{Tea}\langle i \rangle$, 
  and $\texttt{Water}\langle i \rangle$ to encode the remaining drinks.

We are using the angular brackets "$\langle$" and "$\rangle$" as part of the variable names 
because our parser for propositional logic accepts these symbols as part of variable names.  The parser would be confused if we would use parentheses.

## Importing the Necessary Modules

Our goal is to solve this puzzle by first coding it as a solvability problem of propositional logic and then to solve the resulting set of clauses using the algorithm of Davis and Putnam.

In [ ]:
import { solve, complement } from './07-Davis-Putnam-JW';

The function `range` takes two natural numbers `start`and `stop`as arguments.  It
returns a set containing the numbers from `start` up to and including the number `stop`. 

In [ ]:
function range(start: number, stop: number): Set<number> {
  return set(...Array.from({ length: stop - start + 1 }, 
                           (_, index) => start + index));
}

In order to be able to transform formulas from propositional logic into sets of clauses we import the module <tt>cnf</tt> which implements the function <tt>normalize</tt> that takes a formula and transforms it into a set of clauses.

In [ ]:
import { normalize, Clause, Literal, Variable, CNF } from './04-CNF'

In order to write formulas conveniently, we use the parser for propositional logic.

In [ ]:
import { parse } from './PropositionalLogicParser'

Using the parser and the module <tt>cnf</tt> we can impement a function $\texttt{parseKNF}(s)$ that takes a string $s$ representing a formula and transforms $s$ into an equivalent set of clauses.

In [ ]:
function parseKNF(s: string): CNF {
    const nestedTuple = parse(s); 
    return normalize(nestedTuple);
}

## Auxiliary Functions

In order to succinctly express the constraints that all houses have different colours, the inhabitants have different nationalities etc., it is convenient to implement a function $\texttt{atMostOne}(V)$ that takes a set of variables $V$ and returns a set of formulas that is true if and only if all the variables from $V$ have different values.

In [ ]:
function atMostOne(S: Set<Variable>): Set<Clause> {
    return S.cartesianProduct(S)
            .filterMap(([p, q]) => p != q,
                       ([p, q]) => set(tpl('¬', p), tpl('¬', q)));
}

Given a name $f$ and an index $i \in\{1,2,3,4,5\}$, the function $\texttt{variable}(i)$ creates the string 
$f\langle i \rangle$, e.g. the call `variable("Japanese", 2)` the following string:
```
Japanese<2>
```

In [ ]:
function variable(f: string, i: number): string {
    return `${f}<${i}>`;
}

In [ ]:
variable("Japanese", 2);

A call of the form $\texttt{somewhere}(x)$ will return a clause that specifies that the person with property $x$ has to live in one of the houses from $1$ to $5$.  In order to be able to insert this clause into a set, we have to make sure that we return a `Set<Variable>`.

In [ ]:
function somewhere(x: string): Set<Literal> {
    return range(1, 5).map(i => variable(x, i))
}

In [ ]:
somewhere('Japanese');

Given an exclusive set of properties $S$ and a house number $i$, the function $\texttt{atMostOne}(S, i)$ returns a set of clauses that specifies that the person living in house number $i$ has at most one of the properties from the set $S$.  For example, if 
$S = \{\texttt{"Japanese"}, \texttt{"Englishman"}, \texttt{"Spaniard"}, \texttt{"Norwegian"}, \texttt{"Ukranian"}\}$, 
then $\texttt{atMostOne}(S, 3)$ specifies that the inhabitant of house number 3 has at most one of the nationalities from the set $S$.

In [ ]:
function atMostOneAt(S: Set<string>, i: number): Set<Clause> {
    return atMostOne(S.map(x => variable(x, i)));
}

In [ ]:
atMostOneAt(set("A", "B", "C"), 1);

The function $\texttt{onePerHouse}(S)$ is called as follows:
$$\texttt{onePerHouse}(\{\texttt{"Japanese"},
       \texttt{"English"}, 
       \texttt{"Spanish"}, \texttt{"Norwegian"}, 
       \texttt{"Ukrainian"}\})
$$
This function creates a set of clauses that expresses that there has to be a house where the Japanese lives, a house where the Englishman lives, a house where the Spaniard lives, a house where the Norwegian lives, and a house
where the Ukranian lives.  Furthermore, the set of clauses would contain clauses that express that these five persons live in **different** houses.

In [ ]:
function onePerHouse(S: Set<Variable>): Set<Clause> {
    const Result = S.map(x => somewhere(x));
    return Result.flatMap(range(1, 5), i => atMostOneAt(S, i));
}

In [ ]:
onePerHouse(set("A", "B"));

Given two properties $a$ and $b$ the function $\texttt{sameHouse}(a, b)$ computes a set of clauses that specifies that if the inhabitant of house number $i$ has the property $a$, then he also has the the property $b$ and vice versa.  For example, $\texttt{sameHouse}(\texttt{"Japanese"}, \texttt{"Dog"})$ specifies that the Japanese guy keeps a dog.

In [ ]:
function sameHouse(a: Variable, b: Variable): Set<Clause> {
    const Result = new Set<Clause>();
    return Result
        .flatMap(range(1, 5),
                 i => parseKNF(`${variable(a, i)} ↔ ${variable(b, i)}`));
}

In [ ]:
sameHouse("Red", "Tea");

Given to properties $a$ and $b$ the function $\texttt{nextTo}(a, b)$ computes a set of clauses that specifies that the inhabitants with properties $a$ and $b$ are direct neighbours.  For example, $\texttt{nextTo}(\texttt{'Japanese'}, \texttt{'Dog'})$ specifies that the Japanese guy lives next to the guy who keeps a dog.

In [ ]:
function nextTo(a: Variable, b: Variable): Set<Clause> {
    const Formulas = [
        `${variable(a, 1)} → ${variable(b, 2)}`, // House 1
        ...[2, 3, 4].map(i => `${variable(a, i)} → ${variable(b, i - 1)} ∨ ${variable(b, i + 1)}`), // Houses 2, 3, 4
        `${variable(a, 5)} → ${variable(b, 4)}`  // House 5
    ];
    const Result = set<Clause>();
    return Result.flatMap(Formulas, formula => parseKNF(formula));
}

In [ ]:
nextTo('A', 'B');

Given to properties $a$ and $b$ the function $\texttt{leftTo}(a, b)$ computes an array of clauses that specifies that the inhabitants with properties $a$ lives in the house to the left of the inhabitant who has property $b$.  For example, $\texttt{livesTo}(\texttt{'Japanese'}, \texttt{'Dog'})$ specifies that the Japanese guy lives in the house to the left of the house where there is a dog.

In [ ]:
function leftTo(a: Variable, b: Variable): Set<Clause> {
    const Formulas = [
        ...[1, 2, 3, 4].map(i => `${variable(a, i)} ↔ ${variable(b, i + 1)}`),
        `¬${variable(a, 5)}`
    ];
    const Result = set<Clause>();
    return Result.flatMap(Formulas, formula => parseKNF(formula));
}

In [ ]:
leftTo('A', 'B');

In [ ]:
const Nations = set("English", "Spanish", "Ukrainian", "Norwegian", "Japanese");
const Drinks  = set("Coffee", "Tea", "Milk", "OrangeJuice", "Water");
const Pets    = set("Dog", "Snails", "Horse", "Fox", "Zebra");
const Brands  = set("LuckyStrike", "Parliaments", "Kools", "Chesterfields", "OldGold");
const Colours = set("Red", "Green", "Ivory", "Yellow", "Blue");

The function `allClauses` returns a set of clauses describing the problem.

In [ ]:
function allClauses(): Set<Clause> {
    const Result = set<Clause>();
    // 1. Base constraints: Every property category gets exactly one house
    Result.flatMap(
        [Nations, Drinks, Pets, Brands, Colours],
        S => onePerHouse(S)
    );
    // 2. The specific puzzle clues
    const puzzleClues = [
        sameHouse("English", "Red"),                  // The Englishman lives in the red house.
        sameHouse("Spanish", "Dog"),                  // The Spaniard owns the dog.
        sameHouse("Coffee", "Green"),                 // Coffee is drunk in the green house.
        sameHouse("Ukrainian", "Tea"),                // The Ukrainian drinks tea.
        leftTo("Ivory", "Green"),                     // The green house is immediately to the right of the ivory house.
        sameHouse("OldGold", "Snails"),               // The Old Gold smoker owns snails.
        sameHouse("Kools", "Yellow"),                 // Kools are smoked in the yellow house.
        parseKNF(variable("Milk", 3)),                // Milk is drunk in the middle house.
        parseKNF(variable("Norwegian", 1)),           // The Norwegian lives in the first house.
        nextTo("Chesterfields", "Fox"),               // Chesterfields smoker lives next to the fox owner.
        nextTo("Kools", "Horse"),                     // Kools are smoked next to the horse.
        sameHouse("LuckyStrike", "OrangeJuice"),      // The Lucky Strike smoker drinks orange juice.
        sameHouse("Japanese", "Parliaments"),         // The Japanese smokes Parliaments.
        nextTo("Norwegian", "Blue")                   // The Norwegian lives next to the blue house.
    ];
    return Result.flatMap(puzzleClues, S => S);
}

In [ ]:
const Clauses = allClauses();
for (const clause of Clauses) {
    console.log(clause);
}

In [ ]:
Clauses.size

In [ ]:
function main(): Set<Clause> {
    return solve(allClauses());
}

Solving the problem takes about 50 milliseconds on my computer.

In [ ]:
console.time("Solution");
const Solution = main();
console.timeEnd("Solution");

In [ ]:
Solution

## Functions to PrettyPrint the Solution

In [ ]:
function arb<T extends Value>(S: Set<T>): T {
    return S.pickRandom();
}

In [ ]:
function extractAssignment(Solution: Set<Clause>): Record<string, number> {
    const Assignment: Record<string, number> = {};
    for (const Unit of Solution) {
        const lit = arb(Unit);
        if (lit !== null && typeof lit === 'string') {
            const litStr = lit;
            const number = parseInt(litStr.slice(-2, -1), 10);
            const name = litStr.slice(0, -3);
            Assignment[name] = number;
        }
    }
    return Assignment;
}

In [ ]:
extractAssignment(Solution);

We need to import the function `display` from `tslab` in order to be able to present the solution graphically. 

In [ ]:
import { display } from 'tslab';

In [ ]:
function showHTML(Solution: Set<Clause>) {
    const Assignment = extractAssignment(Solution);
    let result = '<table style="border:2px solid blue">\n';
    result += '<tr>';
    for (const name of ['House', 'Nationality', 'Drink', 'Animal', 'Brand', 'Colour']) {
        result += '<th style="color:gold; background-color:blue">' + name + '</th>';
    }
    result += '</tr>\n';

    for (let chair = 1; chair <= 5; chair++) {
        result += `<tr><td style="border:1px solid green">${chair}</td>`;
        for (const Class of [Nations, Drinks, Pets, Brands, Colours]) {
            for (const x of Class) {
                if (Assignment[x] === chair) {
                    result += `<td style="border:1px solid green">${x}</td>`;
                }
            }
        }
        result += '</tr>\n';
    }

    result += '</table>';
    display.html(result);
}

In [ ]:
showHTML(Solution);

## Checking the Uniqueness of the Solution

Given a set of unit clauses $UnitClauses$, the function $\texttt{negateSolution}(UnitClauses)$ returns a clause that is the logical negation of $UnitClauses$.

In [ ]:
function negateSolution(UnitClauses: Set<Clause>): Set<Literal> {
    return UnitClauses.map(unit => complement(arb(unit)));
}

In [ ]:
const input = set<Clause>();
const c1 = set<Literal>("a");
const c2 = set<Literal>(complement("b"));
input.add(c1);
input.add(c2);
negateSolution(input);

In [ ]:
function checkUniqueness(Solution: Set<Clause>, Clauses: Set<Clause>): void {
    const negationLiterals = negateSolution(Solution);
    Clauses.add(negationLiterals);
    const alternative = solve(Clauses);
    const isUnsat = alternative.size == 1 && arb(alternative)!.isEmpty();
    if (isUnsat) {
        console.log("The solution is unique!");
    } else {
        console.log("ERROR: The solution is not unique!");
    }
}

In [ ]:
console.time("uniqueness");
checkUniqueness(Solution, Clauses);
console.timeEnd("uniqueness");